# CISC 473: Deep Learning — Project P08
## Abstractive Summarization via Parameter-Efficient Fine-Tuning
**Team:** Choo Wen Bin & Nai Junn Rong  
**Repository:** [https://github.com/wenbin05/DeepLearningP08](https://github.com/wenbin05/DeepLearningP08)

This notebook trains and evaluates **BART-base** under budget-matched LoRA placement configurations on **CNN/DailyMail**.

In [ ]:
# 1. Install dependencies
!pip install -q transformers peft datasets evaluate rouge-score accelerate

In [ ]:
# 2. Parameter Budget Verification
from transformers import BartForConditionalGeneration
from peft import LoraConfig, get_peft_model, TaskType

base_model = BartForConditionalGeneration.from_pretrained('facebook/bart-base')
total_params = sum(p.numel() for p in base_model.parameters())
print(f'Total BART-base parameters: {total_params:,}')

# Budget-matched placement targets (6 layers, rank=8, W_q & W_v)
placements = {
    'Encoder Self-Attn': [f'model.encoder.layers.{i}.self_attn.{p}' for i in range(6) for p in ['q_proj', 'v_proj']],
    'Decoder Self-Attn': [f'model.decoder.layers.{i}.self_attn.{p}' for i in range(6) for p in ['q_proj', 'v_proj']],
    'Cross-Attn': [f'model.decoder.layers.{i}.encoder_attn.{p}' for i in range(6) for p in ['q_proj', 'v_proj']]
}

for name, targets in placements.items():
    m = BartForConditionalGeneration.from_pretrained('facebook/bart-base')
    peft_conf = LoraConfig(task_type=TaskType.SEQ_2_SEQ_LM, r=8, lora_alpha=16, target_modules=targets, bias='none')
    p_model = get_peft_model(m, peft_conf)
    trainable = sum(p.numel() for p in p_model.parameters() if p.requires_grad)
    print(f'-> {name}: {trainable:,} trainable params ({trainable/total_params*100:.4f}%)')
    assert trainable == 147456, 'Budget mismatch!'

In [ ]:
# 3. Run Training with Targeted Placement
# Example running Cross-Attention LoRA
!python src/train.py --placement cross_attn --seed 42 --max_eval_samples 500